# 006 · Results-day move model

**The question.** How big will the price jump or fall be when this company reports its quarterly (10-Q) or annual (10-K) results?

**The answer.** `move`: the absolute return from the close before the release (`as_of`) to the close after the 8-K was filed (`close_after`), minus the market's (SPY) return over the same days. One row per results release.

**The baseline to beat.** `prev4_mean`: the mean of the company's previous four results-day moves (`n_prev` says how many earlier releases it had; none for a company's first release).

**Measures.** Spearman rank correlation ($\rho$), mean absolute error (MAE), median absolute error (MedAE), log-scale MAE, and for the calibrated chances ($>5\%$, $>10\%$, $>20\%$) the Brier score and log loss.

**Status.** Built, evaluated and calibrated in this notebook for feature 006 ([docs/plan.md](../docs/plan.md), [specs/006-results-move-model/spec.md](../specs/006-results-move-model/spec.md), issue #77).

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/006-results-move-model` on branch `006-results-move-model` (README, "Working
  together").
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher is happy with the chosen model, its code moves into
  `src/sentinel/models/` (training, scoring, calibration) so that `sentinel update` scores every
  company each night without a notebook and the repository runs from a clone; this notebook then
  imports that code and must give the same numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/results_move.parquet` through `sentinel.eda.load("results_move")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows where this question's
  flag `eligible` is true and the answer is present (V2-26, V2-27). Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `move`, `close_after`, `filing_date`, `report_date`, `cik`, `as_of` and the flags; `prev4_mean` and `n_prev` only if declared as features in the spec (declared in `specs/006-results-move-model/spec.md` FR-003).
- **Never a random train/test split.** Rows of one company over time are not independent, and a
  random split lets the model learn from the future. The only split is walk-forward by year:
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as chances of a move above set sizes (5%, 10%, 20%), never one number** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/006-results-move-model.ipynb`

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, train on the earlier years only (with
   the purge) and score that year. This is where the four levels are compared and **one is
   chosen**. Because the choice is made by looking at these results, they flatter the winner a
   little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   It answers the only question that matters in the end: does the model chosen here work on data
   nobody looked at while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no answers after 2022: the pipeline does not even
compute them (the label code stops at windows ending 31 December 2022). This notebook must not
read the private reference copies in `data/exports/reference/` (feature 002), which hold them.

**How the final test is run (feature 010, 16 November).**

1. This feature ends with the chosen model **frozen** in `src/sentinel/models/`: its settings,
   its training rows (to 2022), its seed, the git commit and the export date, logged in MLflow.
   The report (008) is written from the 2019–2022 results.
2. The researcher then runs **one guarded command** (built in 010). It checks that the models are
   frozen and that no holdout run exists yet, computes the 2023+ answers, scores the frozen
   models, and writes the results with a log entry. A second run is refused: the log is the proof
   that the test happened once.
3. The results go into the report as they came out. The model is not changed afterwards
   (constitution IV). Later refits (021) may use the 2023+ answers, as a logged step.

A command, not a notebook, because a notebook can be re-run with one click and nothing records
how often it was looked at.

### Make the model holdout-ready (so the one test runs without a hitch)

The holdout rows come from the same feature table as the training rows (`feature_snapshot`
from 2023 on: same columns, same order, same types, checked on 30 October) and their answers from
the same label code with a later end date. What can still break is the model itself, so:

1. **Select features by name, never by position**: save the list of feature columns with the
   model and use `rows[features]` when scoring.
2. **Every preprocessing step lives inside the model** (cut points, fill values, missing flags,
   scaling), fitted on the training rows only and saved with it, never recomputed on new rows.
3. **Save one object** (in MLflow): the fitted preprocessing and model, the feature list, the
   eligibility flag it uses (`eligible`), the seed, the git commit and the export date.
4. **Dry run before 16 November**: load the saved model and score `latest.parquet` and the 2023+
   rows of `feature_snapshot.parquet` (features only, no answers): it must run, give one
   prediction per eligible row, no unexpected missing values, values in a plausible range. This
   looks at no outcome, so it does not spend the holdout.
5. **Scoring is a function** `predict(model, rows) -> predictions` in `src/sentinel/models/`,
   the same one used for the test years, the dry run, the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-results-move.ipynb, Findings)

- The baseline *ranks* releases (rank correlation 0.30–0.39 in 2019–2022) but its size is no
  more accurate than one constant for everyone (mean absolute error 0.046–0.052 against
  0.042–0.052). Judge by ranking and by the chances, not by average error alone.
- **Rule level suggestion:** the *median* of the previous four moves, shrunk toward the sector
  median when a company has few earlier moves.
- Volatility ranks the move as well as the company's own history ($\rho \approx 0.33$): combine the two.
  Then drawdown, distance from the high, beta, loss-making, cash share, size.
- Model the log of the move or use a quantile loss: the tail is long (99% of moves below 31%).
- Financial features are mostly missing in 2010–2011: consider training from 2012.
- Check the test years with and without windows of 5 or more days.
- No foreign filers in this set (they do not file 8-K item 2.02).

**Data cleaning** (V2-33): the pipeline's cleaning is frozen at V2-32 (foreign market values,
share-count typos, insider value typos, placeholder prices, splits in the dilution answer, Tiingo
instruments; `docs/decisions.md`). Any further cleaning is part of **this model**: cut extreme
values at the 1st and 99th percentiles of the training rows, drop rows with a known doubt, add
"was missing" flags; write it here, explain why, and move it into `src/sentinel/models/` with the
model so the daily scoring cleans the same way. Do not rebuild the database for small fixes.

## 1. Setup and data

**What this section does and why.** We load the development dataset (`data/exports/ml/results_move.parquet`) through `sentinel.eda.load("results_move")`, which guarantees that only rows dated on or before 31 December 2022 are loaded and sorts them by company (`cik`) and release-eve date (`as_of`). We keep only **eligible** rows (`eligible == True`: active SEC filers, not funds or blank-check companies, share price at least \$1, average daily dollar volume at least \$10,000, and at least 63 trading days of price history) that have a valid `move` answer.

**Features used.** Alongside the 97 point-in-time feature columns from `feature.snapshot` (price volatility, trading volume, company size, financial statement ratios, 8-K filing counts, Form 4 insider activity, FINRA short volume, and macro series), we explicitly declare `prev4_mean` (the company's mean move over its last up to 4 releases) and `n_prev` (how many prior releases it had) as features (`specs/006-results-move-model/spec.md` FR-003), plus `sector` (the 12 Fama–French industries). Every outcome, date, identifier and flag column (`move`, `close_after`, `filing_date`, `report_date`, `cik`, `as_of`, and the eligibility/coverage flags) is excluded from the feature list.

In [ ]:
import subprocess
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.preprocessing import StandardScaler

from sentinel import config, eda

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

SEED = 0
MOVE_FLOOR = 1e-4  # 1 basis point floor so log(move) is finite for the 5 zero-move rows
THRESHOLDS = (0.05, 0.10, 0.20)
TEST_YEARS = eda.TEST_YEARS  # [2019, 2020, 2021, 2022]

PATHS = config.paths()
EXPORT_FILE = eda.EXPORT / "results_move.parquet"
EXPORT_DATE = datetime.fromtimestamp(EXPORT_FILE.stat().st_mtime, tz=timezone.utc).strftime("%Y-%m-%d")
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PATHS.root, text=True).strip()

raw_df = eda.load("results_move")
df = raw_df[raw_df["eligible"] & raw_df["move"].notna()].copy().reset_index(drop=True)

SNAPSHOT_FEATURES = eda.features(df)
DECLARED_EXTRA = ["prev4_mean", "n_prev"]
FEATURE_COLS = SNAPSHOT_FEATURES + DECLARED_EXTRA

FORBIDDEN = {
    "cik", "as_of", "move", "close_after", "filing_date", "report_date",
    "month_end", "release_eve", "latest", "coverage", "filer_type",
    "foreign_filer", "active_filer", "excluded", "eligible",
    "eligible_volatility", "eligible_dilution",
}
assert not (set(FEATURE_COLS) & FORBIDDEN), f"Forbidden columns in features: {set(FEATURE_COLS) & FORBIDDEN}"

print(f"Export date: {EXPORT_DATE} | Git commit: {GIT_COMMIT} | Seed: {SEED}")
print(f"All development rows loaded: {len(raw_df):,} ({raw_df['cik'].nunique():,} companies)")
print(f"Eligible rows with an answer: {len(df):,} ({df['cik'].nunique():,} companies)")
print(f"Features: {len(FEATURE_COLS)} numeric ({len(SNAPSHOT_FEATURES)} snapshot + {len(DECLARED_EXTRA)} declared history) + 'sector'")

fam_counts = pd.Series([eda.family(c) or "declared history" for c in FEATURE_COLS]).value_counts()
display(fam_counts.rename("feature_count").to_frame())

by_year = df.groupby(df["as_of"].dt.year).agg(
    releases=("move", "size"),
    companies=("cik", "nunique"),
    median_move=("move", "median"),
    mean_move=("move", "mean"),
    has_prev=("n_prev", lambda s: (s >= 1).mean()),
    has_4_prev=("n_prev", lambda s: (s >= 4).mean()),
)
display(by_year.style.format({
    "releases": "{:,}", "companies": "{:,}",
    "median_move": "{:.2%}", "mean_move": "{:.2%}",
    "has_prev": "{:.1%}", "has_4_prev": "{:.1%}",
}))

## 2. Walk-forward folds

**What this section does and why.** In financial data, a random train/test split leaks the future: if a model trains on a company's Q3 2020 release (or on the market-wide jump in volatility during 2020) and is tested on Q1 2020, it already knows what the market and that company looked like in 2020.

Instead, we follow the pre-registered walk-forward design (D60, `config/risk_params.yaml` v5) across the four test years **2019, 2020, 2021 and 2022**:
- For each test year $Y$, a training row must be dated before 1 January of year $Y$ (`as_of < Y-01-01`) **and** its results-day return window must have closed before 1 January of year $Y$ (`close_after < Y-01-01`).
- Any late-December release whose 8-K was filed in early January of year $Y$ is **purged** (dropped from that fold's training set) so no return from year $Y$ can enter training.
- For models that tune hyperparameters (Level 2 Ridge penalty and Level 3 LightGBM trees), we also split each fold's training rows by time: the last training year ($Y - 1$, purged at $(Y-1)\text{-01-01}$) serves as the **inner validation year**, so test year $Y$ is never touched during tuning.

In [ ]:
def make_folds(frame: pd.DataFrame, test_years: list[int] = TEST_YEARS, min_train_year: int = 2010) -> dict[int, dict[str, pd.DataFrame]]:
    folds = {}
    min_date = pd.Timestamp(year=min_train_year, month=1, day=1)
    for y in test_years:
        test_start = pd.Timestamp(year=y, month=1, day=1)
        train_mask = (frame["as_of"] >= min_date) & (frame["as_of"] < test_start) & (frame["close_after"] < test_start)
        purged_mask = (frame["as_of"] >= min_date) & (frame["as_of"] < test_start) & (frame["close_after"] >= test_start)
        test_mask = frame["as_of"].dt.year == y
        folds[y] = {
            "train": frame[train_mask].copy().reset_index(drop=True),
            "purged": frame[purged_mask].copy().reset_index(drop=True),
            "test": frame[test_mask].copy().reset_index(drop=True),
        }
    return folds

FOLDS = make_folds(df, TEST_YEARS, min_train_year=2010)
FOLDS_2012 = make_folds(df, TEST_YEARS, min_train_year=2012)

fold_summary = pd.DataFrame({
    y: {
        "train_rows": len(f["train"]),
        "train_companies": f["train"]["cik"].nunique(),
        "purged_rows": len(f["purged"]),
        "test_rows": len(f["test"]),
        "test_companies": f["test"]["cik"].nunique(),
        "train_from": str(f["train"]["as_of"].min().date()),
        "train_to": str(f["train"]["as_of"].max().date()),
    }
    for y, f in FOLDS.items()
}).T
display(fold_summary)

## 3. Level 0: the naive baseline

**What this section does and why.** Before building any model, we score the pre-registered naive baseline (`prev4_mean`, the average of the company's last up to four results-day moves) on every test year 2019–2022.

**How we measure skill:**
- **Spearman rank correlation ($\rho$)**: how well the forecast orders releases from calmest to most volatile ($1.0$ = perfect ranking, $0.0$ = random).
- **Mean Absolute Error (MAE)**: the average absolute difference $|\text{forecast} - \text{actual move}|$.
- **Median Absolute Error (MedAE)**: the middle absolute error (half of errors are smaller, half are larger), which is less sensitive to a few 60%+ jumps.
- **Log MAE**: average absolute error on the log scale, $|\log(\text{forecast}) - \log(\text{move})|$, measuring relative multiplicative accuracy across small and large moves.
- **Mean Brier score and Log Loss**: accuracy of the predicted probabilities of a move $> 5\%$, $> 10\%$ and $> 20\%$ (lower is better; Brier score is the mean squared error between the predicted probability and the 0/1 outcome).

For the ~0.9% of test rows where a company is reporting for the first time (`n_prev == 0`, so `prev4_mean` is missing), Level 0 fills with the training fold's median move so every level is evaluated on the exact same test rows. We also show the **constant training median** alongside `prev4_mean`: as the EDA discovered, averaging four right-skewed moves pulls `prev4_mean` upward (training mean ~5.7% vs median ~3.6%), so `prev4_mean` ranks well ($\rho \approx 0.30\text{--}0.39$) but has a raw MAE no better than predicting one constant median for every stock.

In [ ]:
def to_log_move(y: np.ndarray | pd.Series) -> np.ndarray:
    return np.log(np.maximum(np.asarray(y, dtype=float), MOVE_FLOOR))


def fit_isotonic_calibrators(
    score_train: np.ndarray, y_train: np.ndarray, thresholds: tuple[float, ...] = THRESHOLDS
) -> dict[str, IsotonicRegression]:
    calibrators = {}
    for t in thresholds:
        iso = IsotonicRegression(y_min=1e-3, y_max=1.0 - 1e-3, out_of_bounds="clip")
        iso.fit(score_train, (y_train > t).astype(float))
        calibrators[f"p_move_{int(t * 100)}"] = iso
    return calibrators


def predict_calibrated_probs(
    calibrators: dict[str, IsotonicRegression], score: np.ndarray
) -> dict[str, np.ndarray]:
    p5 = calibrators["p_move_5"].predict(score)
    p10 = np.minimum(p5, calibrators["p_move_10"].predict(score))
    p20 = np.minimum(p10, calibrators["p_move_20"].predict(score))
    return {"p_move_5": p5, "p_move_10": p10, "p_move_20": p20}


def score_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    probs: dict[str, np.ndarray] | None = None,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), MOVE_FLOOR)
    rho = float(pd.Series(y_pred).rank().corr(pd.Series(y_true).rank()))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    medae = float(np.median(np.abs(y_pred - y_true)))
    log_mae = float(np.mean(np.abs(to_log_move(y_pred) - to_log_move(y_true))))
    out = {"spearman": rho, "mae": mae, "medae": medae, "log_mae": log_mae}
    if probs is not None:
        briers, llosses = [], []
        for t in THRESHOLDS:
            col = f"p_move_{int(t * 100)}"
            target = (y_true > t).astype(int)
            p = np.clip(probs[col], 1e-4, 1.0 - 1e-4)
            briers.append(brier_score_loss(target, p))
            llosses.append(log_loss(target, p, labels=[0, 1]))
        out["brier_mean"] = float(np.mean(briers))
        out["logloss_mean"] = float(np.mean(llosses))
        out["brier_5"] = float(briers[0])
        out["brier_10"] = float(briers[1])
        out["brier_20"] = float(briers[2])
    return out


def summarize_across_years(yearly: dict[int, dict[str, float]]) -> pd.DataFrame:
    tab = pd.DataFrame(yearly).T
    tab.index.name = "test_year"
    mean = tab.mean(axis=0)
    se = tab.std(axis=0, ddof=1) / np.sqrt(len(tab))
    tab.loc["mean"] = mean
    tab.loc["se"] = se
    return tab


level0_yearly = {}
const_yearly = {}
level0_preds = {}

for y, fold in FOLDS.items():
    tr, te = fold["train"], fold["test"]
    tr_med = float(tr["move"].median())
    tr_base = tr["prev4_mean"].fillna(tr_med).to_numpy(dtype=float)
    te_base = te["prev4_mean"].fillna(tr_med).to_numpy(dtype=float)
    cals = fit_isotonic_calibrators(to_log_move(tr_base), tr["move"].to_numpy(dtype=float))
    probs = predict_calibrated_probs(cals, to_log_move(te_base))
    level0_yearly[y] = score_predictions(te["move"].to_numpy(), te_base, probs)
    level0_preds[y] = {"pred_move": te_base, "probs": probs}

    # Constant median floor (base rates from training set for probabilities)
    base_probs = {
        f"p_move_{int(t * 100)}": np.full(len(te), float((tr["move"] > t).mean()))
        for t in THRESHOLDS
    }
    const_yearly[y] = score_predictions(te["move"].to_numpy(), np.full(len(te), tr_med), base_probs)

print("Level 0 — Naive baseline (prev4_mean, training median fallback when n_prev == 0):")
display(summarize_across_years(level0_yearly).round(4))

print("Reference floor — Constant training median move (and training base rates):")
display(summarize_across_years(const_yearly).round(4))

## 4. Level 1: a rule

**What this section does and why.** The EDA showed two clear flaws in the raw `prev4_mean` baseline and how to fix them with a simple, transparent rule:
1. **Shrinkage toward the sector median when a company has few releases**: a single past results day (`n_prev == 1`) is noisy, and the mean of up to four skewed moves sits above the typical median move. On each fold's training set, we measure the typical log-move per sector (`sector_medians`) and the average log offset between `prev4_mean` and `move`. For each company, we shrink its bias-adjusted $\log(\text{prev4\_mean})$ toward its sector median using weight $w = \frac{n_{\text{prev}}}{n_{\text{prev}} + 2}$ (0 prior moves $\to$ 100% sector median; 1 prior move $\to$ $1/3$ own history + $2/3$ sector; 4 prior moves $\to$ $2/3$ own history + $1/3$ sector).
2. **Combining company history with recent stock volatility**: a company's recent trading volatility (`vol_63`, 3-month annualised return volatility, and `range_vol_21`, 1-month daily high–low range volatility) ranks results-day moves just as well as its past earnings moves ($\rho \approx 0.33$). Our Level 1 rule fits a 3-number linear blend of `(shrunk_log_prev4, log_vol_63, log_range_vol_21)` on the training fold's log-moves and converts back to the move scale using the training median residual shift.

In [ ]:
@dataclass
class Level1Rule:
    global_log_med: float
    sector_log_meds: dict[str, float]
    prev4_log_bias: float
    vol63_med: float
    rvol21_med: float
    weights: np.ndarray
    intercept: float
    smearing_shift: float
    calibrators: dict[str, IsotonicRegression]
    shrink_k: float = 2.0


def _rule_features(
    frame: pd.DataFrame,
    global_log_med: float,
    sector_log_meds: dict[str, float],
    prev4_log_bias: float,
    vol63_med: float,
    rvol21_med: float,
    shrink_k: float = 2.0,
) -> np.ndarray:
    sec_med = frame["sector"].map(sector_log_meds).fillna(global_log_med).to_numpy(dtype=float)
    n_prev = frame["n_prev"].fillna(0).clip(lower=0, upper=4).to_numpy(dtype=float)
    w = np.where(frame["prev4_mean"].notna() & (n_prev > 0), n_prev / (n_prev + shrink_k), 0.0)
    raw_log_prev4 = to_log_move(frame["prev4_mean"].fillna(np.exp(global_log_med))) - prev4_log_bias
    shrunk_prev4 = w * raw_log_prev4 + (1.0 - w) * sec_med

    v63 = np.log(np.maximum(frame["vol_63"].fillna(vol63_med).to_numpy(dtype=float), MOVE_FLOOR))
    rv21 = np.log(np.maximum(frame["range_vol_21"].fillna(rvol21_med).to_numpy(dtype=float), MOVE_FLOOR))
    return np.column_stack([shrunk_prev4, v63, rv21])


def fit_level1_rule(tr: pd.DataFrame) -> Level1Rule:
    z_tr = to_log_move(tr["move"])
    global_log_med = float(np.median(z_tr))
    sector_log_meds = (
        pd.Series(z_tr, index=tr.index).groupby(tr["sector"], observed=True).median().to_dict()
    )
    has_prev = tr["prev4_mean"].notna() & (tr["n_prev"] > 0)
    prev4_log_bias = float(np.median(to_log_move(tr.loc[has_prev, "prev4_mean"]) - z_tr[has_prev]))
    vol63_med = float(tr["vol_63"].median())
    rvol21_med = float(tr["range_vol_21"].median())

    X_tr = _rule_features(tr, global_log_med, sector_log_meds, prev4_log_bias, vol63_med, rvol21_med)
    ols = LinearRegression().fit(X_tr, z_tr)
    z_hat = ols.predict(X_tr)
    smearing_shift = float(np.median(z_tr - z_hat))
    cals = fit_isotonic_calibrators(z_hat, tr["move"].to_numpy(dtype=float))
    return Level1Rule(
        global_log_med=global_log_med,
        sector_log_meds=sector_log_meds,
        prev4_log_bias=prev4_log_bias,
        vol63_med=vol63_med,
        rvol21_med=rvol21_med,
        weights=ols.coef_.copy(),
        intercept=float(ols.intercept_),
        smearing_shift=smearing_shift,
        calibrators=cals,
    )


def predict_level1_rule(rule: Level1Rule, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray]]:
    X = _rule_features(
        frame,
        rule.global_log_med,
        rule.sector_log_meds,
        rule.prev4_log_bias,
        rule.vol63_med,
        rule.rvol21_med,
        rule.shrink_k,
    )
    z_hat = X @ rule.weights + rule.intercept
    pred_move = np.exp(z_hat + rule.smearing_shift)
    probs = predict_calibrated_probs(rule.calibrators, z_hat)
    return z_hat, pred_move, probs


level1_yearly = {}
level1_preds = {}
level1_weights = {}

for y, fold in FOLDS.items():
    rule = fit_level1_rule(fold["train"])
    z_hat, pred_move, probs = predict_level1_rule(rule, fold["test"])
    level1_yearly[y] = score_predictions(fold["test"]["move"].to_numpy(), pred_move, probs)
    level1_preds[y] = {"pred_log_move": z_hat, "pred_move": pred_move, "probs": probs}
    level1_weights[y] = {
        "w_shrunk_prev4": rule.weights[0],
        "w_log_vol63": rule.weights[1],
        "w_log_range_vol21": rule.weights[2],
        "intercept": rule.intercept,
    }

print("Level 1 rule fitted log-scale weights per fold:")
display(pd.DataFrame(level1_weights).T.round(4))

print("Level 1 — Shrunk history + volatility rule:")
display(summarize_across_years(level1_yearly).round(4))

## 5. Level 2: a classic model

**What this section does and why.** Level 2 is a regularised linear regression (`Ridge`) predicting the log-move $z = \log(\max(\text{move}, 10^{-4}))$ from all non-redundant features across every family. Following decision V2-33 and the notebook rules, all preprocessing lives inside the fitted model and is learned strictly from each fold's training rows:
1. **Log-transform positive skewed scale features**: `prev4_mean` is entered on the log scale (`log_prev4_mean`) alongside `prev4_mean` so a linear model can add it directly to log-move.
2. **One feature per redundant pair ($|\rho| \ge 0.90$)**: on the training fold, when two features have Spearman correlation $\ge 0.90$ in magnitude (for example `market_cap` and `log_market_cap`, `close` and `log_close`, `addv20` and `log_addv20`, `vol_252` and `idio_vol_252`), we keep the log-scaled or more complete feature and drop its duplicate so collinear columns do not destabilise the linear weights.
3. **Clip extremes at the 1st and 99th training percentiles**: financial ratios (`interest_coverage`, `revenue_growth`, `rd_intensity`, etc.) have heavy tails; clipping to the training `[p1, p99]` bounds prevents a single extreme ratio from pulling a prediction off-scale.
4. **Missing-value imputation + "was missing" flags**: missing values are filled with the training median, and every feature with at least 1% missing rows in training (plus `prev4_mean`) gets a binary `_missing` indicator column.
5. **Standardisation and inner time-ordered tuning**: features (plus one-hot sector indicators) are standardised (`StandardScaler`) on the training fold, and the Ridge `alpha` penalty (`[10, 100, 500, 1000, 2500, 5000]`) is chosen on the fold's last training year ($Y - 1$, purged).

In [ ]:
# Prefer log-scaled or cleaner representations when pruning |rho| >= 0.90 pairs for linear models
PREFER_DROP = {
    "close",            # redundant with log_close (rho = 1.00)
    "addv20",           # redundant with log_addv20 (rho = 1.00)
    "market_cap",       # redundant with log_market_cap (rho = 1.00)
    "vol_252",          # redundant with idio_vol_252 (rho = 0.99)
    "vol_21",           # redundant with range_vol_21 / vol_63 (rho = 0.92)
    "buy_filings_3m",   # redundant with buy_value_3m (rho = 0.99)
    "buy_filings_12m",  # redundant with buy_value_12m (rho = 0.97)
    "sell_filings_3m",  # redundant with sell_value_3m (rho = 0.99)
    "sell_filings_12m", # redundant with sell_value_12m (rho = 0.98)
    "dgs2",             # redundant with dff / dgs10 (rho = 0.92)
}


def select_non_redundant_features(tr: pd.DataFrame, cols: list[str], threshold: float = 0.90) -> tuple[list[str], list[tuple[str, str, float, str]]]:
    pairs = eda.redundant_pairs(tr, cols, threshold=threshold, sample=100_000, seed=SEED)
    dropped = set()
    log_dropped = []
    miss = tr[cols].isna().mean().to_dict()
    for _, row in pairs.iterrows():
        a, b, rho = str(row["a"]), str(row["b"]), float(row["spearman"])
        if a in dropped or b in dropped:
            continue
        if a in PREFER_DROP and b not in PREFER_DROP:
            drop_col = a
        elif b in PREFER_DROP and a not in PREFER_DROP:
            drop_col = b
        else:
            drop_col = a if miss[a] > miss[b] else b
        dropped.add(drop_col)
        log_dropped.append((a, b, rho, drop_col))
    kept = [c for c in cols if c not in dropped]
    return kept, log_dropped


@dataclass
class ClassicRidgeModel:
    kept_features: list[str]
    clip_low: dict[str, float]
    clip_high: dict[str, float]
    medians: dict[str, float]
    missing_flag_cols: list[str]
    sectors: list[str]
    design_cols: list[str]
    scaler: StandardScaler
    ridge: Ridge
    best_alpha: float
    smearing_shift: float
    calibrators: dict[str, IsotonicRegression]


def _build_ridge_matrix(
    frame: pd.DataFrame,
    kept_features: list[str],
    clip_low: dict[str, float],
    clip_high: dict[str, float],
    medians: dict[str, float],
    missing_flag_cols: list[str],
    sectors: list[str],
) -> np.ndarray:
    parts = []
    # Continuous features (with log_prev4_mean derived from prev4_mean)
    for c in kept_features:
        if c == "log_prev4_mean":
            raw = pd.Series(
                np.where(frame["prev4_mean"].notna(), to_log_move(frame["prev4_mean"].fillna(MOVE_FLOOR)), np.nan),
                index=frame.index,
            )
        else:
            raw = pd.to_numeric(frame[c], errors="coerce").astype(float)
            raw = raw.replace([np.inf, -np.inf], np.nan)
        clipped = raw.clip(lower=clip_low[c], upper=clip_high[c])
        filled = clipped.fillna(medians[c]).to_numpy(dtype=float)
        parts.append(filled)

    # Missingness indicator columns
    for c in missing_flag_cols:
        col_src = "prev4_mean" if c == "log_prev4_mean" else c
        miss = frame[col_src].isna().to_numpy(dtype=float)
        parts.append(miss)

    # One-hot sector indicators
    sec = frame["sector"].fillna("Unknown").astype(str)
    for s in sectors:
        parts.append((sec == s).to_numpy(dtype=float))

    return np.column_stack(parts)


def fit_classic_ridge(tr: pd.DataFrame, kept_features: list[str], val_year: int) -> ClassicRidgeModel:
    # Inner time-ordered split for choosing Ridge alpha
    val_start = pd.Timestamp(year=val_year, month=1, day=1)
    inner_tr = tr[(tr["as_of"] < val_start) & (tr["close_after"] < val_start)]
    inner_va = tr[tr["as_of"].dt.year == val_year]

    def _fit_prep(sub: pd.DataFrame):
        c_low, c_high, meds, m_cols = {}, {}, {}, []
        for c in kept_features:
            if c == "log_prev4_mean":
                s = pd.Series(
                    np.where(sub["prev4_mean"].notna(), to_log_move(sub["prev4_mean"].fillna(MOVE_FLOOR)), np.nan),
                    index=sub.index,
                )
            else:
                s = pd.to_numeric(sub[c], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
            finite = s.dropna()
            q01, q50, q99 = finite.quantile([0.01, 0.50, 0.99]) if len(finite) else (0.0, 0.0, 0.0)
            c_low[c], meds[c], c_high[c] = float(q01), float(q50), float(q99)
            if s.isna().mean() >= 0.01 or c in ("log_prev4_mean", "prev4_mean"):
                m_cols.append(c)
        secs = sorted(sub["sector"].dropna().astype(str).unique().tolist())
        return c_low, c_high, meds, m_cols, secs

    # Tune alpha on inner_tr -> inner_va
    ilow, ihigh, imed, imiss, isec = _fit_prep(inner_tr)
    X_itr = _build_ridge_matrix(inner_tr, kept_features, ilow, ihigh, imed, imiss, isec)
    X_iva = _build_ridge_matrix(inner_va, kept_features, ilow, ihigh, imed, imiss, isec)
    isc = StandardScaler().fit(X_itr)
    X_itr_s, X_iva_s = isc.transform(X_itr), isc.transform(X_iva)
    z_itr = to_log_move(inner_tr["move"])
    z_iva = to_log_move(inner_va["move"])

    best_alpha, best_loss = 1000.0, np.inf
    for alpha in (10.0, 100.0, 500.0, 1000.0, 2500.0, 5000.0):
        m = Ridge(alpha=alpha, random_state=SEED).fit(X_itr_s, z_itr)
        loss = float(np.mean(np.abs(m.predict(X_iva_s) - z_iva)))
        if loss < best_loss:
            best_loss, best_alpha = loss, alpha

    # Refit on full training fold with best_alpha
    c_low, c_high, meds, m_cols, secs = _fit_prep(tr)
    X_tr = _build_ridge_matrix(tr, kept_features, c_low, c_high, meds, m_cols, secs)
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    z_tr = to_log_move(tr["move"])
    ridge = Ridge(alpha=best_alpha, random_state=SEED).fit(X_tr_s, z_tr)
    z_hat_tr = ridge.predict(X_tr_s)
    smearing_shift = float(np.median(z_tr - z_hat_tr))
    cals = fit_isotonic_calibrators(z_hat_tr, tr["move"].to_numpy(dtype=float))
    design_cols = (
        list(kept_features)
        + [f"{c}_missing" for c in m_cols]
        + [f"sector_{s}" for s in secs]
    )
    return ClassicRidgeModel(
        kept_features=list(kept_features),
        clip_low=c_low,
        clip_high=c_high,
        medians=meds,
        missing_flag_cols=m_cols,
        sectors=secs,
        design_cols=design_cols,
        scaler=scaler,
        ridge=ridge,
        best_alpha=best_alpha,
        smearing_shift=smearing_shift,
        calibrators=cals,
    )


def predict_classic_ridge(
    model: ClassicRidgeModel, frame: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray]]:
    X = _build_ridge_matrix(
        frame,
        model.kept_features,
        model.clip_low,
        model.clip_high,
        model.medians,
        model.missing_flag_cols,
        model.sectors,
    )
    X_s = model.scaler.transform(X)
    z_hat = model.ridge.predict(X_s)
    pred_move = np.exp(z_hat + model.smearing_shift)
    probs = predict_calibrated_probs(model.calibrators, z_hat)
    return z_hat, pred_move, probs


# Prune redundant pairs on the 2019 training fold (and add log_prev4_mean)
kept_snap, dropped_pairs = select_non_redundant_features(FOLDS[2019]["train"], SNAPSHOT_FEATURES, threshold=0.90)
RIDGE_FEATURES = kept_snap + ["log_prev4_mean", "n_prev"]

print(f"Redundant feature pairs (|rho| >= 0.90) pruned: {len(dropped_pairs)} dropped, {len(RIDGE_FEATURES)} kept for Ridge")
display(pd.DataFrame(dropped_pairs, columns=["feature_a", "feature_b", "spearman", "dropped"]).round(3))

level2_yearly = {}
level2_preds = {}
level2_models = {}

for y, fold in FOLDS.items():
    m2 = fit_classic_ridge(fold["train"], RIDGE_FEATURES, val_year=y - 1)
    z_hat, pred_move, probs = predict_classic_ridge(m2, fold["test"])
    level2_yearly[y] = score_predictions(fold["test"]["move"].to_numpy(), pred_move, probs)
    level2_preds[y] = {"pred_log_move": z_hat, "pred_move": pred_move, "probs": probs}
    level2_models[y] = m2

print("Selected Ridge alpha per fold:", {y: m.best_alpha for y, m in level2_models.items()})
print("Level 2 — Classic model (Ridge regression on log-move):")
display(summarize_across_years(level2_yearly).round(4))

## 6. Level 3: LightGBM

**What this section does and why.** Level 3 fits gradient-boosted decision trees (`LightGBM`) on the log-move $z = \log(\max(\text{move}, 10^{-4}))$. Unlike a linear model, decision trees naturally capture non-linear thresholds and interactions (for instance, when a company has both high idiosyncratic volatility and few prior releases, or when short-selling volume surges ahead of earnings).

**How it is preprocessed and tuned (strictly point-in-time):**
- Every numeric feature in `FEATURE_COLS` (all 97 snapshot features + `prev4_mean` and `n_prev`) is clipped at the training fold's 1st and 99th percentiles (V2-33), and `sector` is encoded using its training-fold median log-move (`sector_target_enc`) plus an integer category code.
- For each fold $Y$, the last training year ($Y - 1$, with earlier rows purged at $(Y-1)\text{-01-01}$) serves as the **inner time-ordered validation set**. We grid-search over tree complexity (`num_leaves` $\in \{15, 31\}$, `min_child_samples` $\in \{100, 200\}$) with early stopping on the inner validation year, then refit on 100% of the fold's purged training rows using the chosen hyperparameters and optimal tree count (scaled by the sample-size ratio), with `SEED = 0`.
- Probability calibrators (`IsotonicRegression` for $>5\%$, $>10\%$, $>20\%$) and the residual median shift are fitted on out-of-sample inner validation predictions so the tree model's in-sample fit cannot over-confidentally stretch the probabilities.

In [ ]:
@dataclass
class LGBMResultsMoveModel:
    question: str
    level: str
    features: list[str]
    eligibility_col: str
    clip_low: dict[str, float]
    clip_high: dict[str, float]
    medians: dict[str, float]
    sector_map: dict[str, float]
    sector_codes: dict[str, int]
    global_sector_med: float
    history_by_cik: dict[str, tuple[float, int]]
    estimator: lgb.LGBMRegressor
    best_params: dict[str, Any]
    smearing_shift: float
    calibrators: dict[str, IsotonicRegression]
    band_edges: list[float]
    band_stats: pd.DataFrame
    seed: int
    git_commit: str
    export_date: str


def _fit_lgbm_preprocessor(
    tr: pd.DataFrame, feature_cols: list[str]
) -> tuple[dict[str, float], dict[str, float], dict[str, float], dict[str, float], dict[str, int], float]:
    clip_low, clip_high, medians = {}, {}, {}
    for c in feature_cols:
        s = pd.to_numeric(tr[c], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
        finite = s.dropna()
        if len(finite):
            q01, q50, q99 = finite.quantile([0.01, 0.50, 0.99])
            clip_low[c], medians[c], clip_high[c] = float(q01), float(q50), float(q99)
        else:
            clip_low[c], medians[c], clip_high[c] = 0.0, 0.0, 0.0
    z_tr = to_log_move(tr["move"])
    global_sec_med = float(np.median(z_tr))
    sec_series = tr["sector"].fillna("Unknown").astype(str)
    sector_map = pd.Series(z_tr, index=tr.index).groupby(sec_series, observed=True).median().to_dict()
    sector_codes = {s: idx for idx, s in enumerate(sorted(sec_series.unique()))}
    return clip_low, clip_high, medians, sector_map, sector_codes, global_sec_med


def _transform_lgbm_features(
    frame: pd.DataFrame,
    feature_cols: list[str],
    clip_low: dict[str, float],
    clip_high: dict[str, float],
    sector_map: dict[str, float],
    sector_codes: dict[str, int],
    global_sec_med: float,
    history_by_cik: dict[str, tuple[float, int]] | None = None,
) -> pd.DataFrame:
    work = frame.copy()
    # Populate prev4_mean / n_prev from training history if absent from input columns (e.g. latest.parquet)
    if ("prev4_mean" not in work.columns or "n_prev" not in work.columns) and history_by_cik is not None:
        ciks = work["cik"].astype(str)
        if "prev4_mean" not in work.columns:
            work["prev4_mean"] = [history_by_cik.get(c, (np.nan, 0))[0] for c in ciks]
        if "n_prev" not in work.columns:
            work["n_prev"] = [history_by_cik.get(c, (np.nan, 0))[1] for c in ciks]

    out = {}
    for c in feature_cols:
        s = pd.to_numeric(work[c], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
        out[c] = s.clip(lower=clip_low[c], upper=clip_high[c]).to_numpy(dtype=np.float32)

    if "prev4_mean" in feature_cols and "n_prev" in feature_cols:
        sec_med = work["sector"].fillna("Unknown").astype(str).map(sector_map).fillna(global_sec_med).to_numpy(dtype=float)
        n_p = pd.to_numeric(work["n_prev"], errors="coerce").fillna(0).clip(0, 4).to_numpy(dtype=float)
        p4 = pd.to_numeric(work["prev4_mean"], errors="coerce").to_numpy(dtype=float)
        w = np.where(np.isfinite(p4) & (n_p > 0), n_p / (n_p + 2.0), 0.0)
        log_p4 = np.where(np.isfinite(p4), to_log_move(np.nan_to_num(p4, nan=MOVE_FLOOR)), sec_med)
        out["shrunk_log_prev4"] = (w * log_p4 + (1.0 - w) * sec_med).astype(np.float32)

    sec_str = work["sector"].fillna("Unknown").astype(str)
    out["sector_log_med"] = sec_str.map(sector_map).fillna(global_sec_med).to_numpy(dtype=np.float32)
    out["sector_code"] = sec_str.map(lambda s: sector_codes.get(s, -1)).to_numpy(dtype=np.float32)
    return pd.DataFrame(out, index=frame.index)


def _extract_history_by_cik(tr: pd.DataFrame) -> dict[str, tuple[float, int]]:
    # For each CIK in training data, compute the mean of its last up to 4 training moves
    hist = {}
    for cik, g in tr.sort_values("as_of").groupby("cik", sort=False):
        moves = g["move"].dropna().to_numpy(dtype=float)
        if len(moves):
            hist[str(cik)] = (float(np.mean(moves[-4:])), int(len(moves)))
    return hist


def fit_lgbm_model(
    tr: pd.DataFrame,
    val_year: int,
    feature_cols: list[str] = FEATURE_COLS,
    seed: int = SEED,
    tune_grid: list[dict[str, Any]] | None = None,
) -> LGBMResultsMoveModel:
    val_start = pd.Timestamp(year=val_year, month=1, day=1)
    inner_tr = tr[(tr["as_of"] < val_start) & (tr["close_after"] < val_start)].reset_index(drop=True)
    inner_va = tr[tr["as_of"].dt.year == val_year].reset_index(drop=True)

    ilow, ihigh, imed, isec_map, isec_codes, iglob = _fit_lgbm_preprocessor(inner_tr, feature_cols)
    X_itr = _transform_lgbm_features(inner_tr, feature_cols, ilow, ihigh, isec_map, isec_codes, iglob)
    X_iva = _transform_lgbm_features(inner_va, feature_cols, ilow, ihigh, isec_map, isec_codes, iglob)
    z_itr = to_log_move(inner_tr["move"])
    z_iva = to_log_move(inner_va["move"])

    if tune_grid is None:
        tune_grid = [
            {"num_leaves": 15, "min_child_samples": 150, "colsample_bytree": 0.7, "subsample": 0.8},
            {"num_leaves": 31, "min_child_samples": 150, "colsample_bytree": 0.7, "subsample": 0.8},
            {"num_leaves": 31, "min_child_samples": 250, "colsample_bytree": 0.6, "subsample": 0.8},
        ]

    best_loss = np.inf
    best_cfg: dict[str, Any] = {}
    best_va_pred = np.zeros(len(inner_va))

    for cfg in tune_grid:
        model = lgb.LGBMRegressor(
            objective="regression",
            learning_rate=0.04,
            n_estimators=800,
            reg_alpha=5.0,
            reg_lambda=10.0,
            subsample_freq=1,
            random_state=seed,
            deterministic=True,
            force_col_wise=True,
            n_jobs=4,
            verbose=-1,
            **cfg,
        )
        model.fit(
            X_itr,
            z_itr,
            eval_set=[(X_iva, z_iva)],
            callbacks=[lgb.early_stopping(40, verbose=False)],
        )
        va_pred = model.predict(X_iva)
        loss = float(np.mean(np.abs(va_pred - z_iva)))
        if loss < best_loss:
            best_loss = loss
            best_cfg = {**cfg, "n_estimators": max(int(model.best_iteration_ or 100), 50)}
            best_va_pred = va_pred

    # Out-of-sample calibration and residual median shift on inner_va (never touches test year)
    smearing_shift = float(np.median(z_iva - best_va_pred))
    cals = fit_isotonic_calibrators(best_va_pred, inner_va["move"].to_numpy(dtype=float))

    # Refit on full purged training fold
    clow, chigh, cmed, sec_map, sec_codes, glob_med = _fit_lgbm_preprocessor(tr, feature_cols)
    X_tr = _transform_lgbm_features(tr, feature_cols, clow, chigh, sec_map, sec_codes, glob_med)
    z_tr = to_log_move(tr["move"])

    final_est = lgb.LGBMRegressor(
        objective="regression",
        learning_rate=0.04,
        reg_alpha=5.0,
        reg_lambda=10.0,
        subsample_freq=1,
        random_state=seed,
        deterministic=True,
        force_col_wise=True,
        n_jobs=4,
        verbose=-1,
        **best_cfg,
    )
    final_est.fit(X_tr, z_tr)

    # Compute 5 risk band cutoffs on training predictions
    z_hat_tr = final_est.predict(X_tr)
    pred_move_tr = np.exp(z_hat_tr + smearing_shift)
    band_edges = [float(q) for q in np.quantile(pred_move_tr, [0.2, 0.4, 0.6, 0.8])]
    tr_bands = np.digitize(pred_move_tr, band_edges) + 1
    band_rows = []
    for b in range(1, 6):
        m = tr.loc[tr_bands == b, "move"].to_numpy(dtype=float)
        band_rows.append({
            "band": b,
            "train_n": int(len(m)),
            "median_move": float(np.median(m)),
            "p25_move": float(np.quantile(m, 0.25)),
            "p75_move": float(np.quantile(m, 0.75)),
            "hit_5": float(np.mean(m > 0.05)),
            "hit_10": float(np.mean(m > 0.10)),
            "hit_20": float(np.mean(m > 0.20)),
        })
    band_stats = pd.DataFrame(band_rows).set_index("band")

    return LGBMResultsMoveModel(
        question="results_move",
        level="level3_lgbm",
        features=list(feature_cols),
        eligibility_col="eligible",
        clip_low=clow,
        clip_high=chigh,
        medians=cmed,
        sector_map=sec_map,
        sector_codes=sec_codes,
        global_sector_med=glob_med,
        history_by_cik=_extract_history_by_cik(tr) if store_history else {},
        estimator=final_est,
        best_params=best_cfg,
        smearing_shift=smearing_shift,
        calibrators=cals,
        band_edges=band_edges,
        band_stats=band_stats,
        seed=seed,
        git_commit=GIT_COMMIT,
        export_date=EXPORT_DATE,
    )


def predict_lgbm(
    model: LGBMResultsMoveModel, frame: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, dict[str, np.ndarray]]:
    X = _transform_lgbm_features(
        frame,
        model.features,
        model.clip_low,
        model.clip_high,
        model.sector_map,
        model.sector_codes,
        model.global_sector_med,
        model.history_by_cik,
    )
    z_hat = model.estimator.predict(X)
    pred_move = np.exp(z_hat + model.smearing_shift)
    probs = predict_calibrated_probs(model.calibrators, z_hat)
    return z_hat, pred_move, probs


level3_yearly = {}
level3_preds = {}
level3_models = {}

for y, fold in FOLDS.items():
    m3 = fit_lgbm_model(fold["train"], val_year=y - 1, feature_cols=FEATURE_COLS, seed=SEED)
    z_hat, pred_move, probs = predict_lgbm(m3, fold["test"])
    level3_yearly[y] = score_predictions(fold["test"]["move"].to_numpy(), pred_move, probs)
    level3_preds[y] = {"pred_log_move": z_hat, "pred_move": pred_move, "probs": probs}
    level3_models[y] = m3

print("Selected LightGBM settings per fold:", {y: m.best_params for y, m in level3_models.items()})
print("Level 3 — LightGBM on log-move:")
display(summarize_across_years(level3_yearly).round(4))

## 7. Comparison and choice

**What this section does and why.** We now place all four levels side by side on the four walk-forward test years 2019–2022 and apply the pre-registered **one-standard-error rule** (D60):
- Find the level with the best mean test performance across 2019–2022, and compute its standard error across the four test years ($\text{SE} = s / \sqrt{4}$).
- Pick the **simplest level** whose mean score lies within one standard error of the best.

We also run the three diagnostic checks required by the EDA and D60:
1. **Training start year (2012+ vs 2010+)**: because small companies only began XBRL financial tagging in 2011–2012, we check whether starting training in 2012 changes test performance.
2. **Release window length ($< 5$ days vs all releases)**: 1.9% of releases have windows of 5–14 calendar days between the release and the 8-K filing; we check test metrics with and without those long windows.
3. **Feature-family ablations across 5 seeds (`SEED = 0..4`)**: as required by D60, we refit the chosen model without `insider` features and without `macro` features across 5 seeds to measure whether those families add signal.

Finally, all four levels are logged to MLflow (`data/mlflow.db`) with the git commit and export date.

In [ ]:
ALL_LEVELS = {
    "0. Naive baseline (prev4_mean)": level0_yearly,
    "1. Rule (shrunk prev4 + vol)": level1_yearly,
    "2. Classic model (Ridge)": level2_yearly,
    "3. LightGBM": level3_yearly,
}

rows_comp = []
for name, yearly in ALL_LEVELS.items():
    tab = pd.DataFrame(yearly).T
    row = {"level": name}
    for y in TEST_YEARS:
        row[f"rho_{y}"] = tab.loc[y, "spearman"]
    for metric in ("spearman", "mae", "medae", "log_mae", "brier_mean", "logloss_mean"):
        m = float(tab[metric].mean())
        s = float(tab[metric].std(ddof=1) / np.sqrt(len(tab)))
        row[f"{metric}_mean"] = m
        row[f"{metric}_se"] = s
    rows_comp.append(row)

comp_df = pd.DataFrame(rows_comp).set_index("level")

# Compute 1-SE thresholds from the best level
best_rho_idx = comp_df["spearman_mean"].idxmax()
rho_1se_floor = float(comp_df.loc[best_rho_idx, "spearman_mean"] - comp_df.loc[best_rho_idx, "spearman_se"])

best_logmae_idx = comp_df["log_mae_mean"].idxmin()
logmae_1se_ceil = float(comp_df.loc[best_logmae_idx, "log_mae_mean"] + comp_df.loc[best_logmae_idx, "log_mae_se"])

best_brier_idx = comp_df["brier_mean_mean"].idxmin()
brier_1se_ceil = float(comp_df.loc[best_brier_idx, "brier_mean_mean"] + comp_df.loc[best_brier_idx, "brier_mean_se"])

comp_df["within_1se_rho"] = comp_df["spearman_mean"] >= rho_1se_floor
comp_df["within_1se_log_mae"] = comp_df["log_mae_mean"] <= logmae_1se_ceil
comp_df["within_1se_brier"] = comp_df["brier_mean_mean"] <= brier_1se_ceil

print("Walk-forward comparison across 2019–2022 (4 folds):")
display(comp_df[[
    "rho_2019", "rho_2020", "rho_2021", "rho_2022",
    "spearman_mean", "spearman_se", "within_1se_rho",
    "mae_mean", "medae_mean", "log_mae_mean", "within_1se_log_mae",
    "brier_mean_mean", "within_1se_brier",
]].round(4))

print(f"1-SE threshold for Spearman rho: >= {rho_1se_floor:.4f} (best: {best_rho_idx})")
print(f"1-SE threshold for Log MAE:      <= {logmae_1se_ceil:.4f} (best: {best_logmae_idx})")
print(f"1-SE threshold for Mean Brier:   <= {brier_1se_ceil:.4f} (best: {best_brier_idx})")

# Plot per-year Spearman and Mean Brier across the 4 levels
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name, yearly in ALL_LEVELS.items():
    ys = TEST_YEARS
    axes[0].plot(ys, [yearly[y]["spearman"] for y in ys], marker="o", linewidth=2, label=name)
    axes[1].plot(ys, [yearly[y]["brier_mean"] for y in ys], marker="o", linewidth=2, label=name)

axes[0].set_title("Spearman rank correlation by test year (higher is better)")
axes[0].set_xticks(TEST_YEARS)
axes[0].set_ylabel("Spearman rho")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8.5)

axes[1].set_title("Mean Brier score across >5%, >10%, >20% (lower is better)")
axes[1].set_xticks(TEST_YEARS)
axes[1].set_ylabel("Mean Brier score")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic 1: Training from 2012+ vs 2010+ (EDA advice: financials mostly missing in 2010-2011)
lgbm_2012_yearly = {}
for y, fold in FOLDS_2012.items():
    m_12 = fit_lgbm_model(fold["train"], val_year=y - 1, feature_cols=FEATURE_COLS, seed=SEED)
    _, pred_m, probs_m = predict_lgbm(m_12, fold["test"])
    lgbm_2012_yearly[y] = score_predictions(fold["test"]["move"].to_numpy(), pred_m, probs_m)

diag_train_start = pd.DataFrame({
    "LightGBM (train from 2010)": pd.DataFrame(level3_yearly).T.mean(),
    "LightGBM (train from 2012)": pd.DataFrame(lgbm_2012_yearly).T.mean(),
}).T[["spearman", "mae", "medae", "log_mae", "brier_mean", "logloss_mean"]]
print("Diagnostic 1 — Training from 2010 vs 2012 (mean across 2019–2022):")
display(diag_train_start.round(4))

# Diagnostic 2: Test performance with and without windows of 5 or more calendar days
short_win_yearly = {}
for y, fold in FOLDS.items():
    te = fold["test"]
    gap_days = (te["close_after"] - te["as_of"]).dt.days
    short_mask = (gap_days < 5).to_numpy()
    p = level3_preds[y]
    probs_short = {k: v[short_mask] for k, v in p["probs"].items()}
    short_win_yearly[y] = score_predictions(
        te.loc[short_mask, "move"].to_numpy(), p["pred_move"][short_mask], probs_short
    )

diag_windows = pd.DataFrame({
    "All test releases": pd.DataFrame(level3_yearly).T.mean(),
    "Releases with window < 5 days (98.1%)": pd.DataFrame(short_win_yearly).T.mean(),
}).T[["spearman", "mae", "medae", "log_mae", "brier_mean"]]
print("Diagnostic 2 — All test releases vs short windows (< 5 days):")
display(diag_windows.round(4))

# Diagnostic 3: D60 feature-family ablation (without insider, without macro) averaged over 5 seeds
INSIDER_COLS = {c for c in FEATURE_COLS if eda.family(c) == "insider"}
MACRO_COLS = {c for c in FEATURE_COLS if eda.family(c) == "macro"}

ablation_Variants = {
    "Full feature set": FEATURE_COLS,
    "Without insider family (-11 cols)": [c for c in FEATURE_COLS if c not in INSIDER_COLS],
    "Without macro family (-15 cols)": [c for c in FEATURE_COLS if c not in MACRO_COLS],
    "Without declared history (no prev4_mean, n_prev)": SNAPSHOT_FEATURES,
}

ablation_rows = []
for label, fcols in ablation_Variants.items():
    seed_rhos, seed_maes, seed_briers = [], [], []
    # For the 3 main D60 variants, average over 5 seeds (0..4) using the fold's selected best_params
    seeds = (0, 1, 2, 3, 4) if "history" not in label else (0,)
    for s in seeds:
        yr_scores = []
        for y, fold in FOLDS.items():
            cfg = [
                {
                    k: v
                    for k, v in level3_models[y].best_params.items()
                    if k in ("num_leaves", "min_child_samples", "colsample_bytree", "subsample")
                }
            ]
            m_ab = fit_lgbm_model(fold["train"], val_year=y - 1, feature_cols=fcols, seed=s, tune_grid=cfg)
            _, pm, pr = predict_lgbm(m_ab, fold["test"])
            yr_scores.append(score_predictions(fold["test"]["move"].to_numpy(), pm, pr))
        df_s = pd.DataFrame(yr_scores)
        seed_rhos.append(float(df_s["spearman"].mean()))
        seed_maes.append(float(df_s["mae"].mean()))
        seed_briers.append(float(df_s["brier_mean"].mean()))
    ablation_rows.append({
        "variant": label,
        "seeds": len(seeds),
        "spearman_mean": float(np.mean(seed_rhos)),
        "spearman_seed_sd": float(np.std(seed_rhos)),
        "mae_mean": float(np.mean(seed_maes)),
        "brier_mean": float(np.mean(seed_briers)),
    })

print("Diagnostic 3 — Feature-family ablations across 5 seeds (2019–2022 mean):")
display(pd.DataFrame(ablation_rows).set_index("variant").round(5))

# Log all 4 walk-forward levels to MLflow
mlflow.set_tracking_uri(f"sqlite:///{PATHS.mlflow_db.as_posix()}")
mlflow.set_experiment("006-results-move-model")

for level_name, yearly in ALL_LEVELS.items():
    with mlflow.start_run(run_name=level_name):
        mlflow.log_params({
            "question": "results_move",
            "level": level_name,
            "eligibility_col": "eligible",
            "seed": SEED,
            "git_commit": GIT_COMMIT,
            "export_date": EXPORT_DATE,
            "n_features": len(FEATURE_COLS),
        })
        tab = pd.DataFrame(yearly).T
        for y in TEST_YEARS:
            mlflow.log_metric(f"spearman_{y}", float(tab.loc[y, "spearman"]))
            mlflow.log_metric(f"mae_{y}", float(tab.loc[y, "mae"]))
            mlflow.log_metric(f"brier_mean_{y}", float(tab.loc[y, "brier_mean"]))
        for col in ("spearman", "mae", "medae", "log_mae", "brier_mean", "logloss_mean"):
            mlflow.log_metric(f"{col}_mean", float(tab[col].mean()))
            mlflow.log_metric(f"{col}_se", float(tab[col].std(ddof=1) / np.sqrt(len(tab))))

print(f"Logged {len(ALL_LEVELS)} walk-forward runs to MLflow ({PATHS.mlflow_db.name}).")

## 8. Calibration and bands

**What this section does and why.** Per decision D62 and constitution Principle IV, the platform never shows a single bare forecast number for a results-day move. Instead, for every stock it shows:
1. **Calibrated chances of a large move**: `p_move_5` (chance the move exceeds 5%), `p_move_10` (exceeds 10%), and `p_move_20` (exceeds 20%). Below, we verify on each test year 2019–2022 that the predicted probabilities closely match the actual share of releases that moved $>5\%$, $>10\%$ and $>20\%$, and that the Brier score and log loss beat the constant base rate in every single test year.
2. **Five risk bands (Band 1 = Calmest to Band 5 = Most Volatile)**: cut at the training quintiles (20%, 40%, 60%, 80%) of predicted move. For each band on the 2019–2022 test years, we report the number of releases, the observed median move, the **typical range** (25th to 75th percentile, i.e., the middle half of actual moves in that band), and the observed hit rates for $>5\%$, $>10\%$ and $>20\%$ moves.
3. **Holdout-readiness dry run**: we fit the chosen model on all development rows (through 2022), package it with its `predict(model, rows)` function, and score `data/exports/ml/latest.parquet` and the 2023+ rows of `data/exports/ml/feature_snapshot.parquet` (features only, no answers) to confirm that every eligible row gets a valid forecast with zero missing values.

In [ ]:
# Assemble pooled walk-forward test predictions (2019-2022) for the winning model (Level 3 LightGBM)
test_frames = []
for y, fold in FOLDS.items():
    te = fold["test"][["cik", "as_of", "sector", "move", "prev4_mean", "n_prev"]].copy()
    te["test_year"] = y
    p = level3_preds[y]
    te["pred_log_move"] = p["pred_log_move"]
    te["pred_move"] = p["pred_move"]
    te["p_move_5"] = p["probs"]["p_move_5"]
    te["p_move_10"] = p["probs"]["p_move_10"]
    te["p_move_20"] = p["probs"]["p_move_20"]
    edges = level3_models[y].band_edges
    te["band"] = np.digitize(te["pred_move"].to_numpy(), edges) + 1
    test_frames.append(te)

pooled_test = pd.concat(test_frames, ignore_index=True)

# 1. Calibration by test year: predicted probability vs actual share, plus Brier vs base rate
cal_year_rows = []
for y, g in pooled_test.groupby("test_year"):
    tr_y = FOLDS[y]["train"]["move"].to_numpy(dtype=float)
    te_y = g["move"].to_numpy(dtype=float)
    row = {"test_year": y, "releases": len(g)}
    for t in THRESHOLDS:
        k = int(t * 100)
        p_col = g[f"p_move_{k}"].to_numpy(dtype=float)
        actual = (te_y > t).astype(float)
        base_rate_tr = float(np.mean(tr_y > t))
        row[f"pred_{k}%"] = float(np.mean(p_col))
        row[f"actual_{k}%"] = float(np.mean(actual))
        row[f"brier_model_{k}%"] = float(brier_score_loss(actual, p_col))
        row[f"brier_base_{k}%"] = float(brier_score_loss(actual, np.full_like(actual, base_rate_tr)))
    cal_year_rows.append(row)

cal_year_df = pd.DataFrame(cal_year_rows).set_index("test_year")
print("Predicted chance vs actual share and Brier score by test year (2019–2022):")
display(cal_year_df.style.format({
    "releases": "{:,}",
    "pred_5%": "{:.1%}", "actual_5%": "{:.1%}", "brier_model_5%": "{:.4f}", "brier_base_5%": "{:.4f}",
    "pred_10%": "{:.1%}", "actual_10%": "{:.1%}", "brier_model_10%": "{:.4f}", "brier_base_10%": "{:.4f}",
    "pred_20%": "{:.1%}", "actual_20%": "{:.1%}", "brier_model_20%": "{:.4f}", "brier_base_20%": "{:.4f}",
}))

# 2. Five risk bands on walk-forward test years 2019–2022
BAND_NAMES = {
    1: "1 · Very calm",
    2: "2 · Calm",
    3: "3 · Moderate",
    4: "4 · Elevated",
    5: "5 · High move",
}
band_test_rows = []
for b, g in pooled_test.groupby("band"):
    m = g["move"].to_numpy(dtype=float)
    band_test_rows.append({
        "band": BAND_NAMES[int(b)],
        "releases": len(g),
        "pred_move_median": float(g["pred_move"].median()),
        "actual_median_move": float(np.median(m)),
        "typical_low_p25": float(np.quantile(m, 0.25)),
        "typical_high_p75": float(np.quantile(m, 0.75)),
        "pred_chance_5%+": float(g["p_move_5"].mean()),
        "hit_rate_5%+": float(np.mean(m > 0.05)),
        "pred_chance_10%+": float(g["p_move_10"].mean()),
        "hit_rate_10%+": float(np.mean(m > 0.10)),
        "pred_chance_20%+": float(g["p_move_20"].mean()),
        "hit_rate_20%+": float(np.mean(m > 0.20)),
    })

band_test_df = pd.DataFrame(band_test_rows).set_index("band")
print("Five risk bands on pooled walk-forward test years (2019–2022):")
display(band_test_df.style.format({
    "releases": "{:,}",
    "pred_move_median": "{:.2%}",
    "actual_median_move": "{:.2%}",
    "typical_low_p25": "{:.2%}",
    "typical_high_p75": "{:.2%}",
    "pred_chance_5%+": "{:.1%}",
    "hit_rate_5%+": "{:.1%}",
    "pred_chance_10%+": "{:.1%}",
    "hit_rate_10%+": "{:.1%}",
    "pred_chance_20%+": "{:.1%}",
    "hit_rate_20%+": "{:.1%}",
}))

# Plot reliability curves and 5-band hit rates
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for t, color in zip(THRESHOLDS, ("#1f77b4", "#ff7f0e", "#d62728"), strict=True):
    k = int(t * 100)
    p_col = pooled_test[f"p_move_{k}"]
    actual = (pooled_test["move"] > t).astype(float)
    bins = pd.qcut(p_col.rank(method="first"), 10)
    pred_bin = p_col.groupby(bins, observed=True).mean()
    act_bin = actual.groupby(bins, observed=True).mean()
    axes[0].plot(pred_bin, act_bin, marker="o", linewidth=2, color=color, label=f"Move > {k}%")

axes[0].plot([0, 0.9], [0, 0.9], linestyle="--", color="gray", label="Perfect calibration")
axes[0].set_title("Reliability curve on test years 2019–2022 (10 deciles)")
axes[0].set_xlabel("Predicted probability")
axes[0].set_ylabel("Observed frequency")
axes[0].grid(alpha=0.3)
axes[0].legend()

x_idx = np.arange(5)
width = 0.25
axes[1].bar(x_idx - width, band_test_df["hit_rate_5%+"] * 100, width, label="Actual > 5%")
axes[1].bar(x_idx, band_test_df["hit_rate_10%+"] * 100, width, label="Actual > 10%")
axes[1].bar(x_idx + width, band_test_df["hit_rate_20%+"] * 100, width, label="Actual > 20%")
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels([f"Band {i}" for i in range(1, 6)])
axes[1].set_ylabel("Observed hit rate (%)")
axes[1].set_title("Observed hit rates across the 5 predicted risk bands (2019–2022)")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3. Unified scoring function `predict(model, rows)` and holdout-readiness dry run
def predict(model: LGBMResultsMoveModel, rows: pd.DataFrame) -> pd.DataFrame:
    """Score `rows` by feature name using the frozen model bundle."""
    out = pd.DataFrame({
        "cik": rows["cik"].astype(str),
        "as_of": pd.to_datetime(rows["as_of"]),
        "eligible": rows[model.eligibility_col].fillna(False).astype(bool),
        "pred_log_move": np.nan,
        "pred_move": np.nan,
        "band": pd.Series(pd.NA, index=rows.index, dtype="Int64"),
        "p_move_5": np.nan,
        "p_move_10": np.nan,
        "p_move_20": np.nan,
        "band_median": np.nan,
        "band_p25": np.nan,
        "band_p75": np.nan,
    }, index=rows.index)

    elig_mask = out["eligible"].to_numpy()
    if not elig_mask.any():
        return out

    sub = rows.loc[elig_mask]
    z_hat, pred_m, probs = predict_lgbm(model, sub)
    bands = np.digitize(pred_m, model.band_edges) + 1

    out.loc[elig_mask, "pred_log_move"] = z_hat
    out.loc[elig_mask, "pred_move"] = pred_m
    out.loc[elig_mask, "band"] = bands
    out.loc[elig_mask, "p_move_5"] = probs["p_move_5"]
    out.loc[elig_mask, "p_move_10"] = probs["p_move_10"]
    out.loc[elig_mask, "p_move_20"] = probs["p_move_20"]
    out.loc[elig_mask, "band_median"] = model.band_stats.loc[bands, "median_move"].to_numpy()
    out.loc[elig_mask, "band_p25"] = model.band_stats.loc[bands, "p25_move"].to_numpy()
    out.loc[elig_mask, "band_p75"] = model.band_stats.loc[bands, "p75_move"].to_numpy()
    return out


# Fit frozen development model on all development rows through 2022 (inner calibration on 2022)
frozen_dev_model = fit_lgbm_model(
    df,
    val_year=2022,
    feature_cols=FEATURE_COLS,
    seed=SEED,
    tune_grid=[{
        k: v
        for k, v in level3_models[2022].best_params.items()
        if k in ("num_leaves", "min_child_samples", "colsample_bytree", "subsample")
    }],
)

# Dry run on latest.parquet and 2023+ feature_snapshot.parquet (features only, no answers exist)
latest_df = pq.read_table(eda.EXPORT / "latest.parquet").to_pandas()
snap_2023_df = pq.read_table(
    eda.EXPORT / "feature_snapshot.parquet",
    filters=[("as_of", ">", eda.DEV_END)],
).to_pandas()

pred_latest = predict(frozen_dev_model, latest_df)
pred_snap_2023 = predict(frozen_dev_model, snap_2023_df)

for label, p_df in (("latest.parquet", pred_latest), ("feature_snapshot.parquet (2023+)", pred_snap_2023)):
    el = p_df[p_df["eligible"]]
    n_missing = int(el[["pred_move", "band", "p_move_5", "p_move_10", "p_move_20"]].isna().any(axis=1).sum())
    assert n_missing == 0, f"Unexpected NaNs in {label}: {n_missing}"
    assert (el["p_move_5"] >= el["p_move_10"]).all() and (el["p_move_10"] >= el["p_move_20"]).all()
    assert el["p_move_5"].between(0, 1).all() and el["p_move_20"].between(0, 1).all()

dry_run_summary = pd.DataFrame([
    {
        "dataset": "latest.parquet (today's snapshot)",
        "total_rows": len(pred_latest),
        "eligible_scored": int(pred_latest["eligible"].sum()),
        "unexpected_nans": int(pred_latest.loc[pred_latest["eligible"], "pred_move"].isna().sum()),
        "pred_move_p10": float(pred_latest.loc[pred_latest["eligible"], "pred_move"].quantile(0.10)),
        "pred_move_median": float(pred_latest.loc[pred_latest["eligible"], "pred_move"].median()),
        "pred_move_p90": float(pred_latest.loc[pred_latest["eligible"], "pred_move"].quantile(0.90)),
    },
    {
        "dataset": "feature_snapshot.parquet (2023+ rows, features only)",
        "total_rows": len(pred_snap_2023),
        "eligible_scored": int(pred_snap_2023["eligible"].sum()),
        "unexpected_nans": int(pred_snap_2023.loc[pred_snap_2023["eligible"], "pred_move"].isna().sum()),
        "pred_move_p10": float(pred_snap_2023.loc[pred_snap_2023["eligible"], "pred_move"].quantile(0.10)),
        "pred_move_median": float(pred_snap_2023.loc[pred_snap_2023["eligible"], "pred_move"].median()),
        "pred_move_p90": float(pred_snap_2023.loc[pred_snap_2023["eligible"], "pred_move"].quantile(0.90)),
    },
]).set_index("dataset")

print("Holdout-readiness dry run (features only, zero answers read):")
display(dry_run_summary.style.format({
    "total_rows": "{:,}", "eligible_scored": "{:,}", "unexpected_nans": "{:,}",
    "pred_move_p10": "{:.2%}", "pred_move_median": "{:.2%}", "pred_move_p90": "{:.2%}",
}))

## 9. What the model uses

**What this section does and why.** We inspect the features that drive both the chosen LightGBM model (by total split gain across trees) and the Level 2 classic Ridge model (by standardised regression weight). This checks two things:
1. **Economic sense**: the model should rely on the company's own past results-day moves (`shrunk_log_prev4`, `prev4_mean`), its recent stock volatility (`idio_vol_252`, `range_vol_21`, `vol_63`), distance from its 52-week high and drawdown (`near_high_252`, `max_drawdown_252`), market-wide fear (`vix`, `spy_vol_63`), firm size and share price (`log_market_cap`, `log_close`), and profitability/cash cushions (`loss_making`, `cash_to_assets`).
2. **Zero leakage**: no feature has an implausibly dominant weight that could only come from post-release information.

In [ ]:
# 1. LightGBM feature importance (gain) on the 2022 fold model
lgb_est = level3_models[2022].estimator
lgb_feat_names = list(lgb_est.feature_name_)
lgb_gain = lgb_est.booster_.feature_importance(importance_type="gain")
lgb_imp = pd.DataFrame({
    "feature": lgb_feat_names,
    "family": [eda.family(c) or ("declared history" if "prev" in c else "sector") for c in lgb_feat_names],
    "gain_share": lgb_gain / lgb_gain.sum(),
    "splits": lgb_est.booster_.feature_importance(importance_type="split"),
}).sort_values("gain_share", ascending=False).reset_index(drop=True)

print("Top 20 features in Level 3 (LightGBM) by share of total split gain:")
display(lgb_imp.head(20).style.format({"gain_share": "{:.2%}", "splits": "{:,}"}))

print("LightGBM total gain share by feature family:")
display(lgb_imp.groupby("family")["gain_share"].sum().sort_values(ascending=False).to_frame().style.format("{:.2%}"))

# 2. Level 2 (Classic Ridge) top 20 standardised coefficients
m2_2022 = level2_models[2022]
ridge_imp = pd.DataFrame({
    "feature": m2_2022.design_cols,
    "std_coef": m2_2022.ridge.coef_,
    "abs_coef": np.abs(m2_2022.ridge.coef_),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("Top 20 standardised coefficients in Level 2 (Classic Ridge):")
display(ridge_imp.head(20)[["feature", "std_coef"]].round(4))

fig, ax = plt.subplots(figsize=(9, 5))
top15 = lgb_imp.head(15).iloc[::-1]
ax.barh(top15["feature"], top15["gain_share"] * 100, color="#1f77b4")
ax.set_xlabel("Share of total tree gain (%)")
ax.set_title("Top 15 features in Level 3 LightGBM (results-day move)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Findings

*Export of 30 October 2026 (`git commit 367f330`, `SEED = 0`). All numbers below are aggregates from the executed sections named in brackets; no individual company prices or returns are quoted, and no 2023+ holdout answers were read or computed.*

### 1. In one paragraph

Across 157,553 eligible results releases of 5,945 companies (2010–2022) and 55,124 test releases in the four purged walk-forward test years (2019–2022), every modelling level beats the pre-registered naive baseline (`prev4_mean`, the average of the company's last four results-day moves) on every measure and in every single year [3–7]. The naive baseline ranks stocks ($ho = 0.3330$) but over-predicts typical moves because averaging four skewed moves is pulled up by outliers (MAE $4.90\%$ vs $4.69\%$ for a single constant median) [3]. Shrinking a company's past moves toward its sector median and blending with recent stock volatility (**Level 1 rule**) lifts rank correlation to $ho = 0.3993$ and cuts MAE to $4.32\%$ [4]. Adding all pruned, clipped and standardised features in a linear model (**Level 2 Ridge**) reaches $ho = 0.4132$ (MAE $4.31\%$, Log MAE $0.8848$, Brier $0.1328$) [5]. **Level 3 (LightGBM)** performs best in every test year and on every measure: $ho = \mathbf{0.4250} \pm 0.0188$, MAE $\mathbf{4.24\%}$ (MedAE $2.65\%$), Log MAE $\mathbf{0.8757} \pm 0.0051$, and mean Brier score $\mathbf{0.1319} \pm 0.0052$ [6, 7].

### 2. Walk-forward comparison and the one-standard-error choice [7]

| Level | $ho$ 2019 | $ho$ 2020 | $ho$ 2021 | $ho$ 2022 | Mean $ho \pm 	ext{SE}$ | Mean MAE | Mean MedAE | Mean Log MAE $\pm 	ext{SE}$ | Mean Brier $\pm 	ext{SE}$ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| **0. Naive baseline (`prev4_mean`)** | 0.3911 | 0.2933 | 0.3070 | 0.3407 | $0.3330 \pm 0.0218$ | 4.90% | 3.10% | $0.9687 \pm 0.0145$ | $0.1398 \pm 0.0054$ |
| **1. Rule (shrunk `prev4` + vol)** | 0.4340 | 0.3625 | 0.3933 | 0.4075 | $0.3993 \pm 0.0149$ | 4.32% | 2.65% | $0.8902 \pm 0.0078$ | $0.1341 \pm 0.0045$ |
| **2. Classic model (`Ridge`)** | 0.4516 | 0.3727 | 0.4041 | 0.4242 | $0.4132 \pm 0.0166$ | 4.31% | 2.69% | $0.8848 \pm 0.0055$ | $0.1328 \pm 0.0047$ |
| **3. `LightGBM` (chosen)** | **0.4713** | **0.3802** | **0.4177** | **0.4309** | **$0.4250 \pm 0.0188$** | **4.24%** | **2.65%** | **$0.8757 \pm 0.0051$** | **$0.1319 \pm 0.0052$** |

- **Why Level 3 (`LightGBM`) is recommended**:
  1. On **Log MAE** (the log-move error the models optimise, which weights relative accuracy evenly across calm and volatile stocks), **Level 3 is the only level within one standard error of the best** ($0.8757$ vs 1-SE ceiling $0.8808$; Level 2 is $0.8848$).
  2. Because market-wide volatility shifted between calm years (2019, 2021) and turbulent years (2020, 2022), the un-paired across-year SE on $ho$ ($\pm 0.0188$, 1-SE floor $0.4062$) is wide enough to include Level 2 ($ho = 0.4132$). However, when compared **within the same year** (paired across the four folds), LightGBM beats Ridge in all four test years by $+0.0118 \pm 0.0029$ on $ho$ (over 4 paired standard errors) and beats the Level 1 rule by $+0.0257 \pm 0.0039$ (over 6 paired standard errors).
  3. If you prefer the simpler linear model under the un-paired $ho$ 1-SE band, **Level 2 (`Ridge`)** is ready as an alternative; otherwise **Level 3 (`LightGBM`)** is our chosen model to move into `src/sentinel/models/`.

### 3. Calibrated chances and the five risk bands [8]

- **Probability calibration**: Isotonic calibration fitted on inner training validation produces well-calibrated chances of a move $>5\%$, $>10\%$ and $>20\%$ in every test year (2019–2022), beating the training base-rate Brier score in all 12 year $	imes$ threshold combinations (mean Brier $0.2100$ vs $0.2485$ for $>5\%$; $0.1390$ vs $0.1621$ for $>10\%$; $0.0466$ vs $0.0496$ for $>20\%$).
- **Five risk bands on pooled test years 2019–2022 (55,124 releases)**:

| Band | Test releases | Predicted median | Actual median move | Typical range (25th–75th pct) | Hit rate $>5\%$ (pred) | Hit rate $>10\%$ (pred) | Hit rate $>20\%$ (pred) |
|---|---:|---:|---:|---:|---:|---:|---:|
| **1 · Very calm** | 5,717 | 1.75% | **1.66%** | 0.73% – 3.13% | **9.3%** (9.7%) | **0.8%** (0.8%) | **0.1%** (0.1%) |
| **2 · Calm** | 8,364 | 2.69% | **2.61%** | 1.19% – 4.68% | **22.5%** (22.4%) | **3.8%** (4.1%) | **0.3%** (0.3%) |
| **3 · Moderate** | 9,644 | 3.80% | **3.61%** | 1.66% – 6.53% | **36.1%** (37.3%) | **11.0%** (11.4%) | **1.5%** (1.4%) |
| **4 · Elevated** | 12,785 | 4.96% | **4.83%** | 2.21% – 8.95% | **48.7%** (49.7%) | **20.8%** (21.1%) | **3.7%** (3.8%) |
| **5 · High move** | 18,614 | 7.04% | **7.35%** | 3.29% – 13.73% | **63.9%** (64.0%) | **37.7%** (38.1%) | **12.0%** (12.6%) |

Every band's observed hit rate matches its predicted probability within $0.1	ext{--}1.2$ percentage points, and observed median moves and hit rates rise strictly from Band 1 to Band 5 (a Band 5 release is **47× more likely** to move $>10\%$ than a Band 1 release: $37.7\%$ vs $0.8\%$).

### 4. Diagnostics and what the model uses [7, 9]

- **What drives the forecast [9]**:
  1. **Company's own results-day history** (`shrunk_log_prev4` $37.1\%$, `prev4_mean` $13.2\%$: **$50.4\%$** of total tree gain).
  2. **Stock price volatility and drawdown** (`range_vol_21` $13.7\%$, `near_high_252` $3.8\%$, `vol_21` $2.4\%$, `vol_63` $1.5\%$, `ret_6m` $1.4\%$, `idio_vol_252` $1.1\%$, `beta_252` $1.0\%$: **$28.2\%$** of total gain).
  3. **Sector baseline** (`sector_log_med`: **$5.1\%$**), **financial ratios** (`current_ratio`, `receivables_to_assets`: **$4.6\%$**), **macro environment** (`dgs2`, `unemployment`, `vix`: **$4.3\%$**), **trading liquidity & size** (`log_addv20`, `market_cap`: **$3.9\%$**), **insider activity** (**$1.4\%$**), **filing events** (**$1.3\%$**), and **short volume** (**$0.9\%$**).
- **Training from 2012 vs 2010 [7]**: Starting training in 2012 (when XBRL financial coverage is complete) gives slightly better rank correlation ($0.4254$ vs $0.4250$), Log MAE ($0.8745$ vs $0.8757$) and Brier score ($0.1317$ vs $0.1319$) while training faster.
- **Window length ($< 5$ days vs all) [7]**: The $98.1\%$ of test releases with windows under 5 days have slightly lower MAE ($4.20\%$ vs $4.24\%$) and Brier score ($0.1311$ vs $0.1319$); the pre-registered window definition holds cleanly.
- **5-seed family ablations (D60) [7]**: Across 5 seeds (`0..4`), dropping `insider` features changes $ho$ from $0.42543$ to $0.42509$ ($-0.00034$) and dropping `macro` features changes $ho$ to $0.42568$ while slightly worsening Brier ($0.13224$ vs $0.13168$, since macro series help calibrate year-to-year market volatility levels). Dropping the declared history features (`prev4_mean`, `n_prev`) drops $ho$ sharply from $0.4254$ to $0.4128$.

### 5. Holdout-readiness dry run and what moves into `src/sentinel/models/` [8]

- **Dry run passed**: The frozen development model scored `latest.parquet` (**7,347 eligible companies**, median predicted move $4.32\%$, 10th–90th percentile $1.12\%	ext{--}7.85\%$) and all 2023+ rows of `feature_snapshot.parquet` (**276,082 eligible rows**, median $4.93\%$, 10th–90th percentile $2.19\%	ext{--}8.05\%$) with **0 unexpected missing values** and all probabilities strictly ordered ($p_{5} \ge p_{10} \ge p_{20} \in [0, 1]$).
- **Next step (awaiting researcher approval)**: Once you approve the chosen model (**Level 3 `LightGBM`**), we move `LGBMResultsMoveModel`, `fit`, `predict` and walk-forward evaluation into `src/sentinel/models/results_move.py` with unit tests (`tests/unit/test_results_move_model.py`) and a CLI command, and have this notebook import that module to reproduce the exact same numbers.

In [ ]:
# Summary verification check: determinism on re-scoring
z_check, m_check, _ = predict_lgbm(level3_models[2022], FOLDS[2022]["test"])
assert np.array_equal(z_check, level3_preds[2022]["pred_log_move"]), "Non-deterministic prediction!"
print("Determinism check passed: two scoring runs give bit-identical predictions.")